# 🧠 KTO-only — align on top of your SFT checkpoint

Runs **just KTO** (Kahneman-Tversky Optimization) on top of your saved SFT model,
then scores **SFT (baseline) vs SFT + KTO** on the same 42-item benchmark. Use
this when you want KTO numbers on their own — e.g. because the combined ablation
run skipped KTO for memory. It won't hit that OOM here: **KTO is the only trainer
in the session**, so the GPU isn't fragmented by earlier runs (that fragmentation,
not KTO itself, caused the crash). KTO needs a per-device batch > 1, so it stays at
batch 2 — exactly what your original SFT→DPO→KTO run used successfully.

**What KTO is:** instead of preference *pairs*, KTO learns from **unpaired** binary
feedback — each completion is simply labelled good (desirable) or bad (undesirable).
Here the labels come straight from your DPO pairs: every *chosen* answer is good,
every *rejected* answer is bad.

---

## ▶️ How to run (setup in the right panel, then **Run All**)

1. **Accelerator → GPU** (T4 ×2 or P100) and **Internet → ON**.
2. **+ Add Input → the base model:** `gemma-2-2b-it` (Transformers). *(Not the 9B.)*
3. **+ Add Input → your SFT checkpoint:** attach your training notebook's **Output**
   (it contains the `sft_ckpt` folder). **Required** — KTO trains on top of it.
4. **+ Add Input → your DPO dataset** (the preference-pair JSON files).
5. **Run All.** (~1.5–2 h.) You'll get a `SFT vs SFT+KTO` table, `kto_adapter/`,
   `kto_results.json`, and `kto_table.md` in the Output.

Tune `KTO_MAX_STEPS`, `KTO_DATA_CAP`, `KTO_BETA` in the config cell.


In [1]:
#@title 1 · Install pinned libraries
# ============================================================================
# 0. INSTALL PINNED LIBRARIES  (same versions your training notebook used)
# These exact versions were tested end-to-end (SFT + DPO + KTO + generate).
# ============================================================================
import subprocess, sys
_pkgs = ["transformers==5.16.1", "trl==1.12.0", "peft==0.20.0",
         "datasets==5.0.1", "accelerate==1.14.0", "bitsandbytes"]
print("Installing:", ", ".join(_pkgs))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *_pkgs], check=False)
print("Install step done.")


Installing: transformers==5.16.1, trl==1.12.0, peft==0.20.0, datasets==5.0.1, accelerate==1.14.0, bitsandbytes
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.6 MB/s eta 0:00:00
Install step done.


In [2]:
#@title 2 · Imports + config (KTO settings)
# ============================================================================
# 1. IMPORTS + CONFIG  (KTO-only)
# ----------------------------------------------------------------------------
# Runs ONLY KTO on top of your saved SFT checkpoint, then scores SFT (baseline)
# vs SFT+KTO on the same 42-item benchmark.
# Why this won't OOM (unlike the combined ablation): KTO is the ONLY trainer in
# this session, so the GPU isn't fragmented by earlier runs. (KTO itself REQUIRES
# a per-device batch > 1 to compute its KL term, so we keep batch 2 — the same
# value your original SFT->DPO->KTO run used successfully — and lean on the fresh
# session for the memory headroom.)
# ============================================================================
import os, re, gc, json, glob, time, random, zipfile, collections, datetime, shutil
os.environ["WANDB_DISABLED"]      = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # anti-fragmentation

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import KTOTrainer, KTOConfig

import transformers, trl, peft
print("torch:", torch.__version__, "| transformers:", transformers.__version__,
      "| trl:", trl.__version__, "| peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: no GPU. Set Accelerator -> GPU T4 x2 (or P100) in the right panel.")

random.seed(42); np.random.seed(42); torch.manual_seed(42)

# ============================ KTO SETTINGS ==================================
KTO_LR                 = 5e-6
KTO_BETA               = 0.1     # risk-aversion temperature
KTO_BATCH              = 2       # KTO REQUIRES an actual batch > 1 (needs it for
                                 # the KL term) — 2 is the safe minimum, same as
                                 # your original run.
KTO_GRAD_ACC           = 8       # effective batch = 2 x 8 = 16
KTO_MAX_STEPS          = 60      # KTO is the only trainer here, so give it room (~1.5-2 h)
KTO_DESIRABLE_WEIGHT   = 1.0
KTO_UNDESIRABLE_WEIGHT = 1.0
KTO_DATA_CAP           = 1200    # preference pairs -> 2x labeled examples (good/bad)
# If you ever DO hit an OOM: lower MAX_LENGTH below (e.g. 384) — that is the
# biggest memory lever (attention cost grows with the square of the length).

# ============================ SHARED ======================================
MAX_LENGTH   = 512
EVAL_MAX_GEN = 256
LOAD_IN_4BIT = True
_OPTIM = "paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch"

# harness bookkeeping (holdout off -> no SFT data needed)
HOLDOUT_N = 0; test_sft_raw = []
USE_SELF_CONSISTENCY = False; SC_SAMPLES = 5
SFT_MAX_STEPS = 150; SFT_LR = 2e-4; SFT_BATCH = 4; SFT_GRAD_ACC = 4; LORA_R = 16
DPO_MAX_STEPS = KTO_MAX_STEPS; DPO_LR = KTO_LR; DPO_BETA = KTO_BETA

OUT_DIR = "/kaggle/working"; os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- REASONING SYSTEM PROMPT (same as training) ----------------
SYSTEM_PROMPT = """You are a careful, logic-focused reasoning assistant.
Solve the task one step at a time, then give a single final answer.

Rules:
- Work through the problem with one operation per line; keep each step short.
- Follow every stated constraint exactly. If the task is impossible or breaks a
  constraint, say so instead of guessing.
- Double-check your result before you write the final answer.
- Do not repeat these instructions or the question back.

Reply using this structure and nothing else:
<reasoning>
your step-by-step working, one operation per line
</reasoning>
<answer>
the final result only — a number, or TRUE/FALSE, or a short phrase, with no extra words
</answer>"""

def build_prompt(question):
    return (f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{question}<end_of_turn>\n"
            f"<start_of_turn>model\n")

assert KTO_BATCH >= 2, "KTO needs an actual per-device batch >= 2 (it uses it for the KL term)."
print("\nKTO config ready. batch=%d x grad_acc=%d (eff %d) | steps=%d | data cap=%d pairs"
      % (KTO_BATCH, KTO_GRAD_ACC, KTO_BATCH*KTO_GRAD_ACC, KTO_MAX_STEPS, KTO_DATA_CAP))


torch: 2.10.0+cu128 | transformers: 5.16.1 | trl: 1.12.0 | peft: 0.20.0
CUDA available: True
GPU: Tesla T4 | 15.6 GB

KTO config ready. batch=2 x grad_acc=8 (eff 16) | steps=60 | data cap=1200 pairs


In [3]:
#@title 3 · Find base model + SFT checkpoint
# ============================================================================
# 2. FIND THE INPUTS under /kaggle/input
#    - the base gemma-2-2b-it model (required)
#    - your SFT checkpoint sft_ckpt (REQUIRED here — KTO builds on top of it)
# ============================================================================
def _walk_for(pred):
    for root, _dirs, files in os.walk("/kaggle/input"):
        if pred(root, files):
            return root
    return None

# base model = full HF checkpoint (config + safetensors), NOT an adapter, NOT the 9B
BASE_PATH = _walk_for(lambda r, fs:
    "config.json" in fs and any(x.endswith(".safetensors") for x in fs)
    and "adapter_config.json" not in fs and "9b" not in r.lower())

# SFT checkpoint = a LoRA adapter. Prefer a folder literally named sft_ckpt.
SFT_CKPT = _walk_for(lambda r, fs:
    r.rstrip("/").endswith("sft_ckpt") and "adapter_config.json" in fs) \
    or _walk_for(lambda r, fs:
    "adapter_config.json" in fs and any(x.startswith("adapter_model.") for x in fs))

print("BASE model    :", BASE_PATH)
print("SFT checkpoint:", SFT_CKPT)
assert BASE_PATH, ("Base model not found. + Add Input -> the gemma-2-2b-it model "
                   "(Transformers). Do NOT rely on the 9B here.")
assert SFT_CKPT, ("SFT checkpoint not found. + Add Input -> your training notebook's "
                  "OUTPUT (it contains the sft_ckpt folder). KTO trains on top of it.")

tokenizer = AutoTokenizer.from_pretrained(SFT_CKPT)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")


BASE model    : /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2
SFT checkpoint: /kaggle/input/notebooks/hifitaekn/sft-dpo-kto-and-more/sft_ckpt
Tokenizer loaded.


In [4]:
#@title 4 · Load preference data -> KTO good/bad
# ============================================================================
# 3. LOAD PREFERENCE DATA  -> build the KTO (unpaired good/bad) dataset
# Every "chosen" completion becomes a desirable example (label=True) and every
# "rejected" one an undesirable example (label=False).
# ============================================================================
_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/_extracted"]

def _extract_any_zips():
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        out = os.path.join("/kaggle/working/_extracted",
                           os.path.splitext(os.path.basename(z))[0])
        os.makedirs(out, exist_ok=True)
        try:
            with zipfile.ZipFile(z) as zf: zf.extractall(out)
        except Exception as e:
            print(f"  (could not extract {z}: {e})")

def _find_file(fname):
    for root in _SEARCH_ROOTS:
        if not os.path.isdir(root): continue
        hits = glob.glob(os.path.join(root, "**", fname), recursive=True)
        if hits: return sorted(hits, key=len)[0]
    return None

def load_json_files(filenames, label="dataset"):
    _extract_any_zips()
    data, missing = [], []
    for fname in filenames:
        p = _find_file(fname)
        if p is None: missing.append(fname); continue
        with open(p) as f: data.extend(json.load(f))
        print(f"  loaded from {p}")
    if missing: print(f"  NOTE: {label} file(s) not found: {missing}")
    return data

print("Loading preference datasets...")
dpo_files = ["CodingFinal2.json","basic_science_dataset1_dpo.json","dpo_others.json","sft_dpo_2.json"]
all_dpo = load_json_files(dpo_files, "DPO")
if not all_dpo or "chosen" not in all_dpo[0] or "rejected" not in all_dpo[0]:
    tree = glob.glob("/kaggle/input/**/*.json", recursive=True)[:60]
    raise FileNotFoundError(
        "No preference data with chosen/rejected found. Attach your DPO dataset.\n"
        "JSON files currently visible:\n  " + "\n  ".join(tree))
random.seed(42); random.shuffle(all_dpo)
train_dpo_raw = all_dpo[:KTO_DATA_CAP]

# ---- build the KTO dataset (prompt, completion, label) ----
kto_rows = []
for ex in train_dpo_raw:
    p = build_prompt(ex["prompt"])
    kto_rows.append({"prompt": p, "completion": ex["chosen"].strip(),   "label": True})
    kto_rows.append({"prompt": p, "completion": ex["rejected"].strip(), "label": False})
random.seed(42); random.shuffle(kto_rows)
kto_train_ds = Dataset.from_list(kto_rows)
n_good = sum(1 for r in kto_rows if r["label"])
print(f"KTO dataset: {len(kto_rows)} labeled examples "
      f"({n_good} desirable / {len(kto_rows)-n_good} undesirable) "
      f"from {len(train_dpo_raw)} pairs")


Loading preference datasets...
  loaded from /kaggle/input/datasets/hifitaekn/tunix-dpo-dataset/CodingFinal2.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-dpo-dataset/basic_science_dataset1_dpo.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-dpo-dataset/dpo_others.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-dpo-dataset/sft_dpo_2.json
KTO dataset: 2400 labeled examples (1200 desirable / 1200 undesirable) from 1200 pairs


In [5]:
#@title 5 · Scoring harness (same benchmark as training)
# ============================================================================
# 4. EVALUATION HARNESS  (real correctness scoring + benchmark + evaluator)
# ============================================================================
from fractions import Fraction

def extract_answer(text):
    if text is None: return None
    m = re.search(r'<answer>\s*(.*?)\s*</answer>', text, re.DOTALL | re.IGNORECASE)
    if m: return m.group(1).strip()
    t = text.strip()
    m = re.search(r'(?:final\s+answer|answer)\s*[:\-]\s*(.+)', t, re.IGNORECASE)
    if m: return m.group(1).strip().splitlines()[0].strip()
    m = re.search(r'\b(TRUE|FALSE|YES|NO)\b', t, re.IGNORECASE)
    if m: return m.group(1)
    nums = re.findall(r'[-\u2212]?\d[\d,]*(?:\.\d+)?\s*%?', t)
    if nums: return nums[-1].strip()
    lines = [ln.strip() for ln in t.splitlines() if ln.strip()]
    return lines[-1] if lines else None

def check_format(r):
    if r is None: return False, False
    low = r.lower()
    return ("<reasoning>" in low and "</reasoning>" in low,
            "<answer>" in low and "</answer>" in low)

def count_steps(r):
    if not r: return 0
    m = re.search(r'<reasoning>(.*?)</reasoning>', r, re.DOTALL | re.IGNORECASE)
    body = m.group(1) if m else r
    return len([ln for ln in body.splitlines() if ln.strip()])

def has_verify(r):
    return bool(r and re.search(r'\b(verif|check|confirm|re-?check|validate|sanity)\w*', r, re.IGNORECASE))

_BOOL_T={"true","t","yes","y","correct","valid"}; _BOOL_F={"false","f","no","n","incorrect","invalid","impossible"}
def _clean(s): return "" if s is None else str(s).replace('\u2212','-').strip().strip('.').strip()
def to_number(s):
    if s is None: return None
    s=str(s).replace('\u2212','-').replace(',','').replace('$','').replace('\u20ac','').replace('\u00a3','').strip()
    s=s.replace('%','').strip()
    m=re.fullmatch(r'\s*(-?\d+)\s*/\s*(-?\d+)\s*', s)
    if m:
        try: return float(Fraction(int(m.group(1)),int(m.group(2))))
        except ZeroDivisionError: return None
    mm=re.findall(r'-?\d[\d]*\.?\d*(?:[eE][-+]?\d+)?', s)
    if not mm: return None
    try: return float(mm[-1])
    except ValueError: return None
def to_bool(s):
    c=re.sub(r'[^a-z]','',_clean(s).lower())
    if c in _BOOL_T: return True
    if c in _BOOL_F: return False
    low=_clean(s).lower()
    if re.search(r'\b(true|yes|correct)\b',low): return True
    if re.search(r'\b(false|no|incorrect|impossible)\b',low): return False
    return None
def infer_type(exp):
    if to_bool(exp) is not None and re.fullmatch(r'[^a-zA-Z0-9]*(true|false|yes|no)[^a-zA-Z0-9]*',_clean(exp).lower()):
        return "bool"
    if to_number(exp) is not None: return "number"
    return "string"
def is_correct(exp, gen, atype=None, tol=1e-2):
    if exp is None or gen is None: return False
    atype = atype or infer_type(exp)
    if atype=="bool":
        eb,gb=to_bool(exp),to_bool(gen); return (eb is not None) and eb==gb
    if atype=="number":
        en,gn=to_number(exp),to_number(gen)
        return en is not None and gn is not None and abs(en-gn)<=max(tol,abs(en)*1e-3)
    e=re.sub(r'[^a-z0-9 ]','',_clean(exp).lower()).strip()
    g=re.sub(r'[^a-z0-9 ]','',_clean(gen).lower()).strip()
    if not e: return False
    if e==g: return True
    if len(e)<=40 and e in g: return True
    et,gt=set(e.split()),set(g.split())
    return bool(et) and len(et&gt)/len(et)>=0.8
def _vote_key(a):
    if a is None: return None
    n=to_number(a)
    if n is not None: return f"num:{round(n,6)}"
    b=to_bool(a)
    if b is not None: return f"bool:{b}"
    return "str:"+re.sub(r'[^a-z0-9]','',_clean(a).lower())

# ---------- curated ground-truth benchmark (balanced + verified) ----------
# 42 items, evenly spread across 5 reasoning categories. Every numeric/bool
# answer was computed/verified in Python, so the benchmark is trustworthy.
BENCHMARK = [
 {"id":"a1","category":"arithmetic","type":"number","question":"What is the value of (-18)^2 - 2^3?","answer":"316"},
 {"id":"a2","category":"arithmetic","type":"number","question":"Compute 7 + 6 * 5 - 4 / 2.","answer":"35"},
 {"id":"a3","category":"arithmetic","type":"number","question":"What is -3^2? (interpret as the negative of three squared)","answer":"-9"},
 {"id":"a4","category":"arithmetic","type":"number","question":"Evaluate 100 - 3 * (2 + 4)^2.","answer":"-8"},
 {"id":"a5","category":"arithmetic","type":"number","question":"What is 15% of 240?","answer":"36"},
 {"id":"a6","category":"arithmetic","type":"number","question":"Compute the average of 12, 18, 27, and 43.","answer":"25"},
 {"id":"a7","category":"arithmetic","type":"number","question":"What is 2^10?","answer":"1024"},
 {"id":"a8","category":"arithmetic","type":"number","question":"Evaluate 144 / 12 + 9 * 3.","answer":"39"},
 {"id":"a9","category":"arithmetic","type":"number","question":"What is 40% of 350?","answer":"140"},
 {"id":"w1","category":"word_problem","type":"number","question":"A train travels 60 km in 45 minutes. What is its average speed in km/h?","answer":"80"},
 {"id":"w2","category":"word_problem","type":"number","question":"Sarah has 3 boxes with 12 pencils each. She gives away 8 pencils. How many pencils remain?","answer":"28"},
 {"id":"w3","category":"word_problem","type":"number","question":"A shirt costs $40. It is discounted 25%, then 8% sales tax is added. Final price in dollars?","answer":"32.4"},
 {"id":"w4","category":"word_problem","type":"number","question":"A tank holds 240 liters. A pipe fills it at 8 liters per minute. How many minutes to fill it?","answer":"30"},
 {"id":"w5","category":"word_problem","type":"number","question":"John is twice as old as Mary. In 5 years their ages sum to 40. How old is Mary now?","answer":"10"},
 {"id":"w6","category":"word_problem","type":"number","question":"A rectangle is 8 cm by 5 cm. What is its perimeter in cm?","answer":"26"},
 {"id":"w7","category":"word_problem","type":"number","question":"You buy 3 items at $4.50 each and pay with a $20 bill. How much change in dollars?","answer":"6.5"},
 {"id":"w8","category":"word_problem","type":"number","question":"A car uses 6 liters of fuel per 100 km. How many liters for a 250 km trip?","answer":"15"},
 {"id":"w9","category":"word_problem","type":"number","question":"A book has 300 pages. You read 20 pages a day. How many days to finish?","answer":"15"},
 {"id":"l1","category":"logic_tf","type":"bool","question":"All roses are flowers. Some flowers fade quickly. Does it follow that some roses fade quickly? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"l2","category":"logic_tf","type":"bool","question":"If it is raining then the ground is wet. The ground is wet. Does it necessarily follow that it is raining? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"l3","category":"logic_tf","type":"bool","question":"All cats are mammals. All mammals are animals. Therefore all cats are animals. Is this valid? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l4","category":"logic_tf","type":"bool","question":"No fish are birds. All sparrows are birds. Therefore no sparrows are fish. Is this valid? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l5","category":"logic_tf","type":"bool","question":"If A > B and B > C, then A > C. Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l6","category":"logic_tf","type":"bool","question":"Every student passed the exam. Tom is a student. Therefore Tom passed the exam. Is this valid? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l7","category":"logic_tf","type":"bool","question":"Some birds cannot fly. Penguins are birds. Therefore penguins cannot fly. Is this a valid deduction? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"l8","category":"logic_tf","type":"bool","question":"If today is Monday, then tomorrow is Tuesday. Today is Monday. Is tomorrow Tuesday? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s1","category":"science_tf","type":"bool","question":"Is water made of hydrogen and oxygen? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s2","category":"science_tf","type":"bool","question":"Does sound travel faster in air than in water? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"s3","category":"science_tf","type":"bool","question":"Is the boiling point of water at sea level 100 degrees Celsius? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s4","category":"science_tf","type":"bool","question":"Are spiders insects? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"s5","category":"science_tf","type":"bool","question":"Is the sun a star? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s6","category":"science_tf","type":"bool","question":"Does the human heart have four chambers? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s7","category":"science_tf","type":"bool","question":"Is the Earth the third planet from the Sun? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s8","category":"science_tf","type":"bool","question":"Do plants release oxygen during photosynthesis? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"f1","category":"short_factual","type":"string","question":"What is the chemical symbol for gold? Give only the symbol.","answer":"Au"},
 {"id":"f2","category":"short_factual","type":"string","question":"What planet is known as the Red Planet? One word.","answer":"Mars"},
 {"id":"f3","category":"short_factual","type":"string","question":"How many sides does a hexagon have? Give the number.","answer":"6"},
 {"id":"f4","category":"short_factual","type":"string","question":"What is the capital of Japan? One word.","answer":"Tokyo"},
 {"id":"f5","category":"short_factual","type":"string","question":"What gas do humans breathe in to survive? One word.","answer":"Oxygen"},
 {"id":"f6","category":"short_factual","type":"string","question":"How many continents are there on Earth? Give the number.","answer":"7"},
 {"id":"f7","category":"short_factual","type":"string","question":"What is the chemical symbol for sodium? Give only the symbol.","answer":"Na"},
 {"id":"f8","category":"short_factual","type":"string","question":"How many degrees are in a right angle? Give the number.","answer":"90"},
]

def build_holdout(test_rows, n):
    """Hold out a slice of your OWN data (unseen in training) as a generalization
    check. Only items with a clean numeric/bool answer are kept so scoring is exact."""
    items=[]
    for i,ex in enumerate(test_rows):
        a=extract_answer(ex.get("chosen",""))
        if a is None: continue
        t=infer_type(a)
        if t in ("number","bool"):
            items.append({"id":f"h{i}","category":"held_out","question":ex["prompt"],"answer":a,"type":t})
        if len(items)>=n: break
    return items

HOLDOUT = build_holdout(test_sft_raw, HOLDOUT_N)
EVAL_SET = BENCHMARK + HOLDOUT
print(f"Evaluation set: {len(EVAL_SET)} items ({len(BENCHMARK)} curated + {len(HOLDOUT)} held-out)")

RESULTS = {
 "meta":{"model":"google/gemma-2-2b-it","generated_at":datetime.datetime.now(datetime.timezone.utc).isoformat(),
         "eval_n":len(EVAL_SET),"n_curated":len(BENCHMARK),"n_holdout":len(HOLDOUT),
         "categories":sorted(set(x["category"] for x in EVAL_SET)),"framework":"pytorch-trl-qlora","device":"gpu"},
 "config":{"sft":{"max_steps":SFT_MAX_STEPS,"lr":SFT_LR,"effective_batch":SFT_BATCH*SFT_GRAD_ACC,"lora_r":LORA_R},
           "dpo":{"max_steps":DPO_MAX_STEPS,"lr":DPO_LR,"beta":DPO_BETA}},
 "stages":[], "training":{}}
ALL_RECORDS = {}   # stage_id -> full per-item records (for the teacher/judge pass)

@torch.no_grad()
def _gen(m, prompt, do_sample, temperature=0.7, top_k=40, anti_repeat=False):
    """Generate one completion.
    anti_repeat=True  -> only for the UNTRAINED base model, which otherwise falls
                         into degenerate loops. Adds repetition controls.
    anti_repeat=False -> clean greedy decoding for the trained SFT/DPO models. This
                         is critical: repetition penalties / n-gram bans corrupt the
                         structured <reasoning>...</reasoning><answer>...</answer>
                         output (they block the repeated tag tokens), which tanks
                         both format and answer accuracy. Trained models don't loop,
                         so they need no such crutch."""
    m.eval()
    prev = m.config.use_cache; m.config.use_cache = True
    ids = tokenizer(prompt, return_tensors="pt").to(m.device)
    kw = dict(max_new_tokens=EVAL_MAX_GEN, pad_token_id=tokenizer.pad_token_id, do_sample=do_sample)
    if anti_repeat:
        kw.update(repetition_penalty=1.3, no_repeat_ngram_size=3)
    if do_sample:
        kw.update(temperature=temperature, top_k=top_k)
    out = m.generate(**ids, **kw)
    m.config.use_cache = prev
    txt = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)
    for stop in ["<end_of_turn>","<eos>","<pad>"]:
        if stop in txt: txt = txt.split(stop)[0]
    return txt.strip()

def evaluate_model(m, stage_id, stage_name, self_consistency=0, anti_repeat=False, record=True):
    print("\n" + "="*64 + f"\nEVALUATING: {stage_name}\n" + "="*64)
    t0=time.time(); records=[]
    for j,it in enumerate(EVAL_SET):
        prompt=build_prompt(it["question"])
        if self_consistency and self_consistency>1:
            cand=[extract_answer(_gen(m,prompt,True,anti_repeat=anti_repeat)) for _ in range(self_consistency)]
            keys=[_vote_key(a) for a in cand if a is not None]
            if keys:
                win=collections.Counter(keys).most_common(1)[0][0]
                ans=next(a for a in cand if a is not None and _vote_key(a)==win)
            else: ans=None
            # one clean greedy pass so format/steps reflect a real generation
            # (the ANSWER still comes from the majority vote above)
            resp=_gen(m,prompt,False,anti_repeat=anti_repeat)
        else:
            resp=_gen(m,prompt,False,anti_repeat=anti_repeat); ans=extract_answer(resp)
        hr,ha=check_format(resp)
        records.append({"id":it["id"],"category":it["category"],"question":it["question"],
            "expected":it["answer"],"generated":ans,
            "correct":bool(is_correct(it["answer"],ans,it.get("type"))),
            "format_ok":bool(hr and ha),"steps":count_steps(resp),"verify":has_verify(resp),
            "response":resp[:1500]})
        if (j+1)%10==0: print(f"  ...{j+1}/{len(EVAL_SET)}")
    n=len(records); cn=sum(r["correct"] for r in records); fn=sum(r["format_ok"] for r in records)
    overall={"n":n,"correct":cn,"format_correct":fn,
        "answer_accuracy":round(cn/n*100,1),"format_accuracy":round(fn/n*100,1),
        "avg_steps":round(sum(r["steps"] for r in records)/n,2),
        "verify_rate":round(sum(r["verify"] for r in records)/n*100,1)}
    by_cat={}
    for c in sorted(set(r["category"] for r in records)):
        rs=[r for r in records if r["category"]==c]; cc=sum(r["correct"] for r in rs)
        by_cat[c]={"n":len(rs),"correct":cc,"accuracy":round(cc/len(rs)*100,1)}
    stage={"id":stage_id,"name":stage_name,"order":len(RESULTS["stages"]),
        "self_consistency":self_consistency,"overall":overall,"by_category":by_cat,
        "samples":[{k:r[k] for k in ("question","category","expected","generated","correct","format_ok","response")} for r in records[:8]]}
    if record:                        # ablation runs pass record=False to stay
        RESULTS["stages"].append(stage)   # out of the main base->sft->dpo->kto story
        ALL_RECORDS[stage_id] = records   # kept in memory for the teacher/judge pass
    print(f"  Answer accuracy: {overall['answer_accuracy']}%  ({cn}/{n})")
    print(f"  Format accuracy: {overall['format_accuracy']}%  ({fn}/{n})")
    for c,v in by_cat.items(): print(f"     {c:14} {v['accuracy']:5.1f}%  ({v['correct']}/{v['n']})")
    print(f"  (eval time: {(time.time()-t0)/60:.1f} min)")
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return stage

print("Harness ready.")


Evaluation set: 42 items (42 curated + 0 held-out)
Harness ready.


In [6]:
#@title 6 · Run KTO · score SFT vs SFT+KTO · save
# ============================================================================
# 5. RUN KTO  (score SFT baseline -> train KTO -> score SFT+KTO -> save)
# ============================================================================
bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True) \
      if (LOAD_IN_4BIT and torch.cuda.is_available()) else None

def load_fresh_sft(trainable):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(
        BASE_PATH, quantization_config=bnb,
        device_map={"": 0} if torch.cuda.is_available() else None,
        torch_dtype=torch.bfloat16, attn_implementation="eager")
    return PeftModel.from_pretrained(base, SFT_CKPT, is_trainable=trainable)

# ---------------- (1) SFT baseline (the starting point) ----------------
print("\n" + "#"*64 + "\n# BASELINE — the SFT checkpoint\n" + "#"*64)
m0 = load_fresh_sft(False); m0.config.use_cache = True
evaluate_model(m0, "sft", "SFT (baseline)")
del m0
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# ---------------- (2) KTO ----------------
KTO_DONE = False
try:
    print("\n" + "#"*64 + "\n# KTO — from the SFT checkpoint\n" + "#"*64)
    m = load_fresh_sft(True); m.config.use_cache = False
    kargs = KTOConfig(
        output_dir=f"{OUT_DIR}/kto", per_device_train_batch_size=KTO_BATCH,
        gradient_accumulation_steps=KTO_GRAD_ACC, num_train_epochs=1, max_steps=KTO_MAX_STEPS,
        learning_rate=KTO_LR, beta=KTO_BETA,
        desirable_weight=KTO_DESIRABLE_WEIGHT, undesirable_weight=KTO_UNDESIRABLE_WEIGHT,
        lr_scheduler_type="cosine", warmup_steps=10, logging_steps=10,
        save_strategy="no", report_to=[], max_length=MAX_LENGTH,
        precompute_ref_log_probs=True, per_device_eval_batch_size=1,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        bf16=torch.cuda.is_available(), optim=_OPTIM)
    t0 = time.time()
    ktr = KTOTrainer(model=m, ref_model=None, args=kargs,
                     train_dataset=kto_train_ds, processing_class=tokenizer)
    ktr.train()
    RESULTS["training"]["kto_minutes"] = round((time.time()-t0)/60, 2)
    KTO_ADAPTER = f"{OUT_DIR}/kto_adapter"
    ktr.model.save_pretrained(KTO_ADAPTER); tokenizer.save_pretrained(KTO_ADAPTER)
    print(f"KTO done in {RESULTS['training']['kto_minutes']} min, adapter saved to {KTO_ADAPTER}")
    evaluate_model(ktr.model, "kto", "SFT + KTO")
    KTO_DONE = True
    del ktr, m
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nKTO skipped:", type(e).__name__, e)
    print("If this is an OOM: lower MAX_LENGTH (e.g. 384) in the config cell — that")
    print("is the biggest memory lever. (Keep KTO_BATCH at 2; KTO needs batch > 1.)")

# ---------------- (3) compare + save ----------------
def _stage(sid): return next((s for s in RESULTS["stages"] if s["id"] == sid), None)
sft_s, kto_s = _stage("sft"), _stage("kto")
if sft_s and kto_s:
    sa = sft_s["overall"]["answer_accuracy"]; ka = kto_s["overall"]["answer_accuracy"]
    sf = sft_s["overall"]["format_accuracy"]; kf = kto_s["overall"]["format_accuracy"]
    delta = ka - sa
    print("\n" + "="*52)
    print("KTO RESULT  (from the SFT checkpoint)")
    print("="*52)
    print(f"{'Stage':<18}{'Answer':>10}{'Format':>10}")
    print("-"*52)
    print(f"{'SFT (baseline)':<18}{sa:>9.1f}%{sf:>9.1f}%")
    print(f"{'SFT + KTO':<18}{ka:>9.1f}%{kf:>9.1f}%")
    print("-"*52)
    verdict = ("KTO HELPED" if delta > 0 else "KTO HURT" if delta < 0 else "KTO made no change")
    print(f"{verdict}: {delta:+.1f} points on answer accuracy.")
    if delta <= 0:
        print("(An honest, report-worthy finding — KTO does not always help on top of DPO.)")
    RESULTS["kto_summary"] = {"sft_accuracy": sa, "kto_accuracy": ka,
                             "delta_points": round(delta, 1), "verdict": verdict,
                             "kto_minutes": RESULTS.get("training", {}).get("kto_minutes")}
    with open(f"{OUT_DIR}/kto_results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, ensure_ascii=False)
    with open(f"{OUT_DIR}/kto_table.md", "w") as f:
        f.write("# KTO vs SFT (from a shared SFT checkpoint)\n\n"
                f"_Benchmark: {RESULTS['meta']['eval_n']} items · "
                f"{len(train_dpo_raw)} preference pairs · {KTO_MAX_STEPS} steps._\n\n"
                "| Stage | Answer acc. | Format acc. |\n|---|---:|---:|\n"
                f"| SFT (baseline) | {sa:.1f}% | {sf:.1f}% |\n"
                f"| SFT + KTO | {ka:.1f}% | {kf:.1f}% |\n\n"
                f"**{verdict}: {delta:+.1f} points.**\n")
    print("\nSaved:  kto_results.json  |  kto_table.md  |  kto_adapter/")
else:
    print("\nKTO did not complete — nothing to compare. See the error above.")



################################################################
# BASELINE — the SFT checkpoint
################################################################


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


EVALUATING: SFT (baseline)
  ...10/42
  ...20/42
  ...30/42
  ...40/42
  Answer accuracy: 83.3%  (35/42)
  Format accuracy: 100.0%  (42/42)
     arithmetic     100.0%  (9/9)
     logic_tf        62.5%  (5/8)
     science_tf      87.5%  (7/8)
     short_factual  100.0%  (8/8)
     word_problem    66.7%  (6/9)
  (eval time: 4.5 min)

################################################################
# KTO — from the SFT checkpoint
################################################################


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

... (4 outputs left out)

Caching reference log probs for train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
10,0.497130
20,0.485645
30,0.469159
40,0.458980
50,0.460677
60,0.456066


KTO done in 185.78 min, adapter saved to /kaggle/working/kto_adapter

EVALUATING: SFT + KTO
  ...10/42
  ...20/42
  ...30/42
  ...40/42
  Answer accuracy: 88.1%  (37/42)
  Format accuracy: 100.0%  (42/42)
     arithmetic     100.0%  (9/9)
     logic_tf        75.0%  (6/8)
     science_tf     100.0%  (8/8)
     short_factual  100.0%  (8/8)
     word_problem    66.7%  (6/9)
  (eval time: 4.7 min)

KTO RESULT  (from the SFT checkpoint)
Stage                 Answer    Format
----------------------------------------------------
SFT (baseline)         83.3%    100.0%
SFT + KTO              88.1%    100.0%
----------------------------------------------------
KTO HELPED: +4.8 points on answer accuracy.

Saved:  kto_results.json  |  kto_table.md  |  kto_adapter/
